<a href="https://colab.research.google.com/github/pirrabu-iitm/dl-genai-project-26-t1/blob/milestone_1/DL_23f3004489_notebook_t12026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jan_2026_dl_gen_ai_project_path = kagglehub.competition_download('jan-2026-dl-gen-ai-project')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
fnames = []
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        fnames.append(filename)
        print(os.path.join(dirname, filename))
        break
        ''

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/LICENSE
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/5-257349-A-15.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00052/drums.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00098/drums.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00028/drums.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00069/drums.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00015/drums.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00000/drums.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/disco/disco.00064/drums.wav
/kaggle/

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # if 'disco' in filename:
        print(os.path.join(dirname, filename))
    if dirname.endswith('00052'):
        break


/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/LICENSE
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/README.md
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50.csv
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50-human.xlsx
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/5-257349-A-15.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/5-195557-A-19.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/2-122820-B-36.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-115920-A-22.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-172649-C-40.wav
/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
 # Make the list of all genres available (alphabetical order)
STEMS = {'drums.wav', 'vocals.wav', 'bass.wav', 'other.wav'} # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0.

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

        # Iterate through Genres
        # Check: if genre folder exists
        # CHECK : Completeness (Does it have all stems?)
        # CHECK : Corruption (Is any file too small? (less than 4kb))
        # size checks
        # Stratified Shuffle Split
     #-------------------------------------------------------------------------

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            pass

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, "genres_stems", genre)

        # Check if genre folder exists
        if not os.path.isdir(genre_path):
            print(f"Skipping missing genre folder: {genre}")
            continue

        valid_songs = []

        # Iterate through songs in genre
        for song_folder in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song_folder)

            if not os.path.isdir(song_path):
                continue

            stem_files = os.listdir(song_path)

            # CHECK 1: Completeness (all stems present)
            if not STEMS.issubset(set(stem_files)):
                continue

            # CHECK 2: Corruption (file size < 4KB)
            corrupted = False
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                if os.path.getsize(stem_path) < 4 * 1024:  # 4KB threshold
                    corrupted = True
                    break

            if corrupted:
                continue

            valid_songs.append(song_path)

        # Stratified Shuffle Split (per genre)
        rng.shuffle(valid_songs)

        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            for song_path in song_list:
                for stem in STEMS:
                    stem_key = stem.replace('.wav', '')
                    stem_path = os.path.join(song_path, stem)
                    target_dict[genre][stem_key].append(stem_path)

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset

In [ ]:
tr, val = build_dataset(DATA_ROOT)

In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= threshold_sec
    """
    records = []

    # ---- COUNT TOTAL FILES ----
    total_files = sum(
        len(files)
        for genre in dataset_dict
        for files in dataset_dict[genre].values()
    )

    print(f"Total files to analyze: {total_files}")

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:

                # ---- Load Audio ----
                y, _ = librosa.load(file_path, sr=sr)
                total_duration = librosa.get_duration(y=y, sr=sr)

                # ---- Find Non-Silent Intervals ----
                intervals = librosa.effects.split(y, top_db=top_db)

                silence_type = []
                max_silence = 0.0

                # ==============================
                # CASE A: Fully Silent
                # ==============================
                if len(intervals) == 0:
                    max_silence = total_duration
                    silence_type.append("Full")

                else:
                    # Convert to seconds
                    intervals_sec = intervals / sr

                    # ==============================
                    # CASE B: START Silence
                    # ==============================
                    start_silence = intervals_sec[0][0]
                    if start_silence > 0:
                        max_silence = max(max_silence, start_silence)
                        silence_type.append("Start")

                    # ==============================
                    # CASE C: END Silence
                    # ==============================
                    end_silence = total_duration - intervals_sec[-1][1]
                    if end_silence > 0:
                        max_silence = max(max_silence, end_silence)
                        silence_type.append("End")

                    # ==============================
                    # CASE D: MIDDLE Silence
                    # ==============================
                    for i in range(len(intervals_sec) - 1):
                        middle_silence = (
                            intervals_sec[i+1][0] - intervals_sec[i][1]
                        )
                        if middle_silence > 0:
                            max_silence = max(max_silence, middle_silence)
                            silence_type.append("Middle")

                # ---- Store Result ----
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(set(silence_type)),
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records)
    return df

In [ ]:
stems_audio = []

try:
    for key in STEM_KEYS:

        # Get file path of selected song stem
        file_path = tr[GENRE_TO_TEST][key][int(SONG_INDEX)]

        # Load audio (Duration 5.0s for speed/consistency)
        y, _ = librosa.load(
            file_path,
            sr=SR,
            duration=DURATION
        )

        stems_audio.append(y)

    print("Audio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [ ]:
# ------------------- write your code here -------------------------------

# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.vstack(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw ** 2))

# Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#-------------------------------------------------------------------------

In [ ]:
corrupted_count = 0
small_count = 0

threshold_corrupt = 4 * 1024
threshold_small = 5.0491 * 1024 * 1024  # in bytes

genres_path = os.path.join(DATA_ROOT, "genres_stems")

for genre in GENRES:
    genre_path = os.path.join(genres_path, genre)
    if not os.path.isdir(genre_path):
        continue

    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue

        for stem_file in os.listdir(song_path):
            if stem_file.endswith(".wav"):
                file_path = os.path.join(song_path, stem_file)
                file_size = os.path.getsize(file_path)

                if file_size < threshold_corrupt:
                    corrupted_count += 1

                if file_size < threshold_small:
                    small_count += 1

total = corrupted_count + small_count

print("Corrupted (<4KB):", corrupted_count)
print("Files <5.0491MB:", small_count)
print("Final Answer:", total)

Corrupted (<4KB): 0
Files <5.0491MB: 1256
Final Answer: 1256


In [ ]:
count_gt = 0   # sounds > 5.0493 MB
count_lt = 0   # sounds < 5.0491 MB

upper_threshold = 5.0493 * 1024 * 1024
lower_threshold = 5.0491 * 1024 * 1024

genres_path = os.path.join(DATA_ROOT, "genres_stems")

for genre in GENRES:
    genre_path = os.path.join(genres_path, genre)
    if not os.path.isdir(genre_path):
        continue

    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue

        for stem_file in os.listdir(song_path):
            if stem_file.endswith(".wav"):
                file_path = os.path.join(song_path, stem_file)
                file_size = os.path.getsize(file_path)

                if file_size > upper_threshold:
                    count_gt += 1
                elif file_size < lower_threshold:
                    count_lt += 1

abs_difference = abs(count_gt - count_lt)

print("Count > 5.0493MB:", count_gt)
print("Count < 5.0491MB:", count_lt)
print("Absolute Difference:", abs_difference)

Count > 5.0493MB: 184
Count < 5.0491MB: 1256
Absolute Difference: 1072


In [ ]:
train_reggae_drums = len(tr["reggae"]["drums"])
val_country_vocals = len(val["country"]["vocals"])

abs_difference = abs(train_reggae_drums - val_country_vocals)

print("Training Reggae Drum Samples:", train_reggae_drums)
print("Validation Country Vocal Samples:", val_country_vocals)
print("Absolute Difference:", abs_difference)

Training Reggae Drum Samples: 83
Validation Country Vocal Samples: 17
Absolute Difference: 66


In [ ]:
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

Total files to analyze: 3320


In [ ]:
total_silent_files = len(df_silence)
print("Total files with silence ≥ 5 seconds:", total_silent_files)

Total files with silence ≥ 5 seconds: 678


In [ ]:
vocals_silence_count = df_silence[df_silence["Stem"] == "vocals"].shape[0]

print("Total number of vocal tracks with silence ≥ 5 secs:", vocals_silence_count)

Total number of vocal tracks with silence ≥ 5 secs: 315


In [ ]:
# Filter for vocals
vocals_df = df_silence[df_silence["Stem"] == "vocals"]

# Compute average Max_Silence_Sec
average_silence_vocals = vocals_df["Max_Silence_Sec"].mean()

print(f"Average silence length in vocals: {average_silence_vocals:.2f} seconds")

Average silence length in vocals: 12.78 seconds


In [ ]:
# Filter for drums in jazz
jazz_drums_silence = df_silence[
    (df_silence["Stem"] == "drums") &
    (df_silence["Genre"] == "jazz")
]

# Count the number of tracks
total_jazz_drums_silence = jazz_drums_silence.shape[0]

print("Total number of jazz drum tracks with silence ≥ 5 secs:", total_jazz_drums_silence)

Total number of jazz drum tracks with silence ≥ 5 secs: 20


In [ ]:
# Filter for jazz drums with silence >=5s
jazz_drums_df = df_silence[
    (df_silence["Stem"] == "drums") &
    (df_silence["Genre"] == "jazz")
]

# Further filter where Silence_Location is exactly "Middle"
jazz_drums_middle_only = jazz_drums_df[
    jazz_drums_df["Silence_Location"].str.strip() == "Middle"
]

# Count the number of tracks
total_jazz_drums_middle_only = jazz_drums_middle_only.shape[0]

print("Total number of jazz drum tracks with silence ≥ 5 secs and location only Middle:",
      total_jazz_drums_middle_only)

Total number of jazz drum tracks with silence ≥ 5 secs and location only Middle: 0


In [ ]:
# Filter for jazz drums with silence >= 5s
jazz_drums_df = df_silence[
    (df_silence["Stem"] == "drums") &
    (df_silence["Genre"] == "jazz")
]

# Further filter for Max_Silence_Sec >= 10
jazz_drums_long_silence = jazz_drums_df[
    jazz_drums_df["Max_Silence_Sec"] >= 10
]

# Count the number of tracks
total_jazz_drums_long_silence = jazz_drums_long_silence.shape[0]

print("Total number of jazz drum tracks with silence ≥ 5 secs and Max_Silence_Sec ≥ 10:",
      total_jazz_drums_long_silence)

Total number of jazz drum tracks with silence ≥ 5 secs and Max_Silence_Sec ≥ 10: 7
